# 04 — Local LLM summary (LM Studio)
Tests `roche_poc.llm`. Requires the LM Studio server running (Developer tab → *Server: Running*) with a model loaded.

Architecture: **pandas computes the profile, the LLM only phrases it**, and every number it cites is checked.

In [ ]:
# Works after `pip install -e .`; the fallback below covers a fresh clone.
import sys, pathlib
_src = pathlib.Path.cwd().resolve().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 160)

## 1. Connection
The base URL must end with `/v1`.

In [ ]:
from roche_poc.llm.client import get_client, is_server_available, list_models
print('Server available:', is_server_available())
list_models()

## 2. The `system` role limitation (documented finding)
Mistral 7B Instruct v0.3 in LM Studio rejects the `system` role (jinja template error, HTTP 400). This is why `build_prompt` merges the instructions into one user message.

In [ ]:
from roche_poc import config
client = get_client()
messages = [
    {'role': 'system', 'content': 'You are a supply chain analyst. Be concise.'},
    {'role': 'user', 'content': 'A supplier has 15 delays and 3 quality issues. One sentence.'},
]
for model in list_models():
    if 'embed' in model:
        continue
    try:
        r = client.chat.completions.create(model=model, messages=messages, max_tokens=80, temperature=0.2)
        print('[OK]  ', model, '->', r.choices[0].message.content.strip())
    except Exception as exc:
        print('[FAIL]', model, '->', str(exc)[:100])

## 3. Profile -> prompt

In [ ]:
from roche_poc.data.build_tables import load_table
from roche_poc.stats import watchlist
from roche_poc.stats.profile import build_profile
from roche_poc.llm.prompts import build_prompt
snapshots = load_table(config.SNAPSHOTS_PATH); comments = load_table(config.COMMENTS_PATH)
vendor = watchlist.rank_entities(snapshots, 'vendor').iloc[0][config.COL_VENDOR]
profile = build_profile(snapshots, comments, 'vendor', vendor)
messages, facts = build_prompt(profile)
print(len(messages[0]['content']), 'characters')
print(messages[0]['content'][:1500])

## 4. Generate and validate

In [ ]:
from roche_poc.llm.summary import generate_summary
result = generate_summary(profile)
print(result.text)
print(result.seconds, 's |', result.validation)

## 5. Evaluation: unsupported-number rate per model
Load each model in LM Studio in turn, run on several entities and compare. Keep the table for the Results chapter.

In [ ]:
import pandas as pd
rows = []
vendors = watchlist.rank_entities(snapshots, 'vendor')[config.COL_VENDOR].head(5)
for v in vendors:
    res = generate_summary(build_profile(snapshots, comments, 'vendor', v))
    rows.append({'vendor': v, 'model': res.model, 'seconds': res.seconds, **{k: res.validation[k] for k in ('unsupported_rate', 'unsupported')}})
pd.DataFrame(rows)